# Trimorious 2.0 | CPU Scientific Playground
Inference only — no GPU, retraining, uploads or changes to model weights. Select **CPU / None** as the hardware accelerator in free Colab. Run the cells through the notebook prompt test, then edit and rerun that prompt cell to test more questions. The optional Gradio cell is for a local Jupyter runtime on your own computer only, not a hosted free Colab runtime. Colab restricts primarily interacting via external web interfaces on free managed runtimes.

CPU 4-bit bitsandbytes inference is slower than T4 and requires a compatible x86-64 CPU, Python wheels, sufficient RAM and disk. A tiny CPU quantization test runs before downloading the full model. No fallback to a full-precision 4B load is attempted if it fails. The exact foundation revision and adapter hashes are retained from your audited release. Comparison mode uses the same prompt and greedy decoding; these are informal tests, not a replacement for GPU benchmark results.

In [ ]:
# 1. Dependencies (fresh runtime recommended)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'peft==0.18.1', 'accelerate==1.15.0',
    'bitsandbytes==0.50.2', 'huggingface_hub==0.36.2', 'safetensors==0.8.0',
    'psutil>=5.9'])
print('Dependencies installed. If Colab requests a restart, restart and run all again.')

In [ ]:
# 2. Configuration and adapter integrity
from pathlib import Path
import hashlib, json, os
try:
    from google.colab import drive
    HOSTED_COLAB = True
except ImportError:
    HOSTED_COLAB = False
if HOSTED_COLAB:
    drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/trimorious_PRIVATE_m1_4_1') if HOSTED_COLAB else Path.cwd()
SOURCE = 'drive' # @param ["drive", "huggingface"]
ADAPTER_PATH = '' # Optional absolute path to final adapter directory
BASE = 'Qwen/Qwen3-4B-Instruct-2507'
REV = 'cdbee75f17c01a7cc42f958dc650907174af0554'
REPO = 'roime81/trimorious-research-4B-v2'
EXPECTED = {'adapter_model.safetensors': '5ce2eadc1ca1f33d28b2a6a517eaad53a73263441c5519c3dc662590514a609d', 'adapter_config.json': '181bb2231889bb9049b78892c907f96103493933055dfb3a4027d393c772dff9'}

def sha(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def matches(p):
    return all((p/name).is_file() and sha(p/name)==digest for name,digest in EXPECTED.items())

if SOURCE == 'drive':
    if ADAPTER_PATH:
        adapter = Path(ADAPTER_PATH)
    else:
        assert PROJECT.is_dir(), 'Project folder not mounted.'
        candidates = sorted({p.parent for p in PROJECT.rglob('adapter_config.json')}, key=lambda p: ('release_package' not in str(p), str(p)))
        adapter = next((p for p in candidates if matches(p)), None)
        assert adapter is not None, 'No matching final adapter in Drive. Set ADAPTER_PATH or SOURCE=huggingface.'
elif SOURCE == 'huggingface':
    from huggingface_hub import snapshot_download, HfApi
    if HOSTED_COLAB:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    else:
        from getpass import getpass
        token = os.environ.get('HF_TOKEN') or getpass('Hugging Face read token: ')
    assert token, 'A read token is required for the private model.'
    commit = HfApi().model_info(REPO, token=token).sha
    adapter = Path(snapshot_download(REPO, revision=commit, token=token, allow_patterns=list(EXPECTED)))
    del token
    print('Hub adapter commit:', commit)
else:
    raise ValueError('Unknown SOURCE')
assert matches(adapter), 'Adapter hashes differ from the audited release. Stop; do not substitute weights.'
config = json.loads((adapter/'adapter_config.json').read_text())
assert config['base_model_name_or_path']==BASE and config['revision']==REV
print('Audited adapter files: verified')


In [ ]:
# 3. CPU preflight and load one 4-bit model + adapter
import torch, psutil, platform, shutil
import bitsandbytes as bnb
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print('CPU:', platform.machine(), '| PyTorch:', torch.__version__, '| bitsandbytes:', bnb.__version__)
assert platform.machine().lower() in ('x86_64','amd64'), 'This notebook targets x86-64 CPU wheels.'
ram = psutil.virtual_memory().available / 2**30
cache = Path(os.environ.get('HF_HOME', str(Path.home()/'.cache/huggingface'))).expanduser()
cache.mkdir(parents=True, exist_ok=True)
disk = shutil.disk_usage(cache).free / 2**30
print(f'Available RAM: {ram:.2f} GiB | Cache disk: {disk:.2f} GiB')
assert ram >= 7, 'Not enough available system RAM for conservative CPU inference. Use a larger-memory runtime or local PC.'
assert disk >= 10, 'Insufficient free disk for downloading foundation weights. Free space first.'
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
# Exercise the CPU 4-bit path before downloading several GB.
try:
    probe = bnb.nn.Linear4bit(64,64,bias=False,compute_dtype=torch.float32,compress_statistics=True,quant_type='nf4').to('cpu')
    with torch.inference_mode():
        probe_out = probe(torch.ones(1,64,dtype=torch.float32))
    assert torch.isfinite(probe_out).all()
    assert probe.weight.quant_state is not None
    del probe, probe_out
except Exception as e:
    raise RuntimeError('CPU NF4 preflight failed. Do not load full-precision weights on this runtime.') from e
print('CPU NF4 preflight passed.')
q = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float32)
tok = AutoTokenizer.from_pretrained(BASE,revision=REV,trust_remote_code=False)
base = AutoModelForCausalLM.from_pretrained(BASE,revision=REV,quantization_config=q,
    device_map={'':'cpu'},torch_dtype=torch.float32,low_cpu_mem_usage=True,trust_remote_code=False)
assert base.is_loaded_in_4bit, 'Expected quantized model.'
model=PeftModel.from_pretrained(base,str(adapter),is_trainable=False).eval()
print('Audited Trimorious adapter loaded on CPU.')

In [ ]:
# 4. Inference functions and smoke test
import time, threading
from contextlib import nullcontext
lock=threading.Lock()

def generate(messages, baseline=False, max_new_tokens=96, temperature=0.0):
    prompt=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=tok(prompt,return_tensors='pt',add_special_tokens=False).to(model.get_input_embeddings().weight.device)
    count=inputs['input_ids'].shape[1]
    if count>1024:
        raise ValueError('Context exceeds 1024 input tokens. Clear chat or shorten your prompt.')
    kwargs=dict(max_new_tokens=int(max_new_tokens),do_sample=temperature>0,pad_token_id=tok.eos_token_id,use_cache=True)
    if temperature>0: kwargs.update(temperature=float(temperature),top_p=0.9)
    with lock, torch.inference_mode(), (model.disable_adapter() if baseline else nullcontext()):
        start=time.perf_counter()
        out=model.generate(**inputs,**kwargs)
        elapsed=time.perf_counter()-start
    answer=tok.decode(out[0,count:],skip_special_tokens=True)
    return answer, f'{out.shape[1]-count} tokens · {elapsed:.2f}s'

answer,stats=generate([{'role':'user','content':'What is 7 multiplied by 8?'}],max_new_tokens=32)
print('Inference smoke test:',answer,stats)
assert answer.strip(), 'Empty generation.'


In [ ]:
# 5. Test any prompt here. Edit and rerun this cell only after loading the model.
PROMPT = 'Explain the difference between correlation and causation using an example.' # @param {type:"string"}
MODE = 'Trimorious 2.0' # @param ["Trimorious 2.0", "Qwen3 foundation", "Compare both"]
MAX_NEW_TOKENS = 96 # @param {type:"integer"}
assert 1 <= MAX_NEW_TOKENS <= 256
messages = [{'role':'user','content':PROMPT}]
if MODE not in ('Trimorious 2.0','Qwen3 foundation','Compare both'):
    raise ValueError('Unknown mode')
for label, is_base in [('Trimorious 2.0',False),('Qwen3 foundation',True)]:
    if MODE not in (label,'Compare both'): continue
    print('\nGenerating:',label,flush=True)
    answer, stats = generate(messages, baseline=is_base, max_new_tokens=MAX_NEW_TOKENS)
    print(answer, '\n',stats,flush=True)

In [ ]:
LAUNCH_LOCAL_GRADIO = False # Set True only in local Jupyter on your own computer.
if LAUNCH_LOCAL_GRADIO and not HOSTED_COLAB:
    # 6. Optional LOCAL Gradio playground (not for hosted Colab)
    import subprocess, sys
    subprocess.check_call([sys.executable,'-m','pip','install','-q','gradio>=6,<7'])
    import gradio as gr
    import getpass, secrets
    os.environ['GRADIO_ANALYTICS_ENABLED']='False'

    def chat(message, history, mode, max_tokens, temperature):
        try:
            if mode=='Compare both (single turn)':
                msgs=[{'role':'user','content':message}]
                v,vs=generate(msgs,False,max_tokens,0.0)
                b,bs=generate(msgs,True,max_tokens,0.0)
                return f'### Trimorious 2.0\n{v}\n\n_{vs}_\n\n---\n### Qwen3 foundation\n{b}\n\n_{bs}_'
            msgs=[]
            for item in history:
                if item.get('role') not in ('user','assistant'): continue
                content=item.get('content')
                if not isinstance(content,str): raise ValueError('Text-only chat. Clear the conversation.')
                # Gradio displays metrics after answers; remove that suffix before reuse.
                if item['role']=='assistant' and '\n\n---\n_' in content:
                    content=content.rsplit('\n\n---\n_',1)[0]
                msgs.append({'role':item['role'],'content':content})
            msgs.append({'role':'user','content':message})
            answer,stats=generate(msgs,mode=='Qwen3 foundation',max_tokens,temperature)
            return answer+'\n\n---\n_'+stats+'_'
        except ValueError as e:
            return str(e)

    password=getpass.getpass('Set a private playground password (at least 12 characters): ')
    assert len(password)>=12, 'Use a stronger password.'
    # Run outside the mounted Drive and block its file-serving path.
    app_dir=Path.home()/'.trimorious_gradio_app'; app_dir.mkdir(exist_ok=True); os.chdir(app_dir)
    with gr.Blocks(title='Trimorious 2.0 | Scientific Playground') as demo:
        gr.Markdown('# Trimorious 2.0 — Scientific Playground\nExperimental research adapter. Verify answers independently. Clear chat before changing models. Comparison mode ignores history and uses greedy decoding.')
        with gr.Row():
            mode=gr.Radio(['Trimorious 2.0','Qwen3 foundation','Compare both (single turn)'],value='Trimorious 2.0',label='Model')
            max_tokens=gr.Slider(32,256,value=96,step=32,label='Maximum response tokens')
            temp=gr.Slider(0,1,value=0,step=0.1,label='Temperature (comparison uses 0)')
        gr.ChatInterface(fn=chat,additional_inputs=[mode,max_tokens,temp],concurrency_limit=1)
    print('Login username: researcher. Use the password you just entered. Open the local URL on this computer.')
    demo.queue(default_concurrency_limit=1).launch(share=False,server_name='127.0.0.1',auth=('researcher',password),blocked_paths=[str(adapter.resolve()),str(PROJECT.resolve())],show_error=False)
else:
    print('Local Gradio not launched. Use the notebook prompt cell.')